# Predicting `target_sales` — a cross-sectional regression notebook

**What this notebook is.** A supervised regression study on 1,000 rows of simulated
financial data. Given a row's `sales` figure and five macro indicators, we predict that
row's `target_sales`.

**What this notebook is not.** A time-series forecast. The dataset has no date, period,
or ordering column, so there is no "next month" to project into. Every claim below is
conditional on knowing the feature values, not on the passage of time. See the caveats
cell at the end before you show any of these numbers to anyone.

**The result you should expect, stated up front so no chart can flatter us later:**
`sales` alone explains almost all of `target_sales`. The headline R² will look
spectacular and it will be almost entirely one feature's doing.

## 1. Setup

Seed is fixed at 42 in every place that consumes randomness — the split, the forests,
the permutation shuffles — so that reruns are comparable. Without this, the
model-ranking table reshuffles between runs and you end up chasing noise.

In [ ]:
import io
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

# Silence only the convergence chatter from Lasso at tiny alphas; we still inspect the
# chosen alpha afterwards, so we are not hiding a fitting failure.
warnings.filterwarnings("ignore", message=".*Objective did not converge.*")

pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

print("Environment ready. Seed =", SEED)

### 1b. Load the data

Tries the local filesystem first (so reruns after a restart don't re-prompt you), then
falls back to the Colab upload widget. The `KNOWN_NAMES` list includes the
browser-mangled `__1_` variant because that is what a re-download usually produces.

In [ ]:
KNOWN_NAMES = [
    "simulated_financial_forecasting_data.csv",
    "simulated_financial_forecasting_data__1_.csv",
]
SEARCH_PATHS = [p for n in KNOWN_NAMES for p in (n, f"/content/{n}", f"./data/{n}")]

df = None
for path in SEARCH_PATHS:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded from disk: {path}")
        break

if df is None:
    try:
        from google.colab import files  # noqa: F401  (only exists inside Colab)

        print("File not found on disk — please upload the CSV.")
        uploaded = files.upload()
        first_key = next(iter(uploaded))
        df = pd.read_csv(io.BytesIO(uploaded[first_key]))
        print(f"Loaded from upload: {first_key}")
    except ImportError as exc:
        raise FileNotFoundError(
            "CSV not found and not running in Colab. Place the file next to this "
            f"notebook under one of these names: {KNOWN_NAMES}"
        ) from exc

TARGET = "target_sales"
FEATURES = [c for c in df.columns if c != TARGET]

print(f"\nShape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Target : {TARGET}")
print(f"Features: {FEATURES}")

**Read this:** 1,000 rows and 7 columns, with `target_sales` held out as the label and
the remaining six as predictors. 1,000 rows is small — enough for linear models to be
stable, thin enough that gradient boosting can memorise noise. Keep that in mind when
the tree models post suspiciously good training scores.

## 2. Exploratory data analysis

Four things worth establishing before modelling: the data is what it claims to be, it is
complete, it isn't secretly duplicated rows, and we can see where the signal lives.

In [ ]:
display(df.head())

In [ ]:
df.info()

In [ ]:
display(df.describe().T)

**Read this:** Every column is float64 with 1,000 non-null entries, so there is no
hidden string contamination and no imputation decision to make. The scales differ by
orders of magnitude — `sales` sits in the thousands, `gdp_growth` around 3 — which is
exactly why scaling goes inside a pipeline later. Nothing in the min/max rows looks like
a sentinel value (`-999`, `0` where zero is impossible), so no cleaning is warranted.

In [ ]:
missing = df.isna().sum()
n_dupes = df.duplicated().sum()

print("Missing values per column:")
print(missing.to_string())
print(f"\nTotal missing cells : {int(missing.sum())}")
print(f"Fully duplicated rows: {int(n_dupes)}")

if missing.sum() == 0 and n_dupes == 0:
    print("\nClean: nothing to impute, nothing to drop.")

**Read this:** No missing cells and no duplicated rows. On real financial data this
would be remarkable and worth distrusting; on simulated data it is expected. It does
mean the train/test split cannot leak via repeated rows, which is a genuine (if
accidental) methodological convenience.

In [ ]:
axes = df.hist(figsize=(13, 8), bins=40, color="#4C72B0", edgecolor="white", grid=False)
for ax in axes.flatten():
    ax.set_title(ax.get_title(), fontsize=10)
plt.suptitle("Distribution of every column", y=1.00, fontsize=13)
plt.tight_layout()
plt.show()

**Read this:** Everything is broadly bell-shaped and unimodal. No log transform is
needed, no heavy right tail of the kind you'd get from real revenue data, no zero-inflation.
This symmetry is itself a tell that the data is simulated — real sales distributions are
almost always skewed.

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(8.5, 6.5))
sns.heatmap(
    corr,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation matrix", fontsize=13)
plt.tight_layout()
plt.show()

target_corr = corr[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)
print("Correlation with the target, strongest first:")
print(target_corr.to_string())

**Read this — this is the central finding of the whole notebook.** `sales` correlates
with `target_sales` at roughly 0.985. The five macro indicators sit near zero, all with
|r| well under 0.05, which is the range you'd get from pure noise at n=1,000. The
hypothesis stated at the top is now confirmed on the data rather than assumed: one
feature carries the signal and five are decoration.

Note also that the macro features barely correlate with *each other*, so there is no
multicollinearity problem to untangle — they are independent noise, not a correlated
bundle whose joint effect is being split.

In [ ]:
plt.figure(figsize=(7.5, 5.5))
sns.regplot(
    data=df,
    x="sales",
    y=TARGET,
    scatter_kws={"alpha": 0.35, "s": 18, "edgecolor": "none"},
    line_kws={"color": "#C44E52", "lw": 2},
)
plt.title(f"sales vs {TARGET}  (r = {df['sales'].corr(df[TARGET]):.4f})", fontsize=12)
plt.tight_layout()
plt.show()

**Read this:** A tight, linear, homoscedastic band. The relationship is not curved and
the spread does not widen with magnitude, which tells us a linear model should be
competitive with anything fancier. If a random forest beats linear regression here by
more than a rounding error, be suspicious of it rather than impressed by it.

## 3. Train / test split

The split happens **before** any scaling, and the scaler lives inside a `Pipeline`. This
ordering matters: fitting a `StandardScaler` on the full dataset would let the test
rows' means and standard deviations bleed into the training transform. The leak is small
in absolute terms but it is the single most common way a notebook quietly overstates its
own accuracy.

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

print(f"Train: {X_train.shape[0]:,} rows")
print(f"Test : {X_test.shape[0]:,} rows")
print(f"\nTarget mean — train {y_train.mean():,.1f} | test {y_test.mean():,.1f}")
print(f"Target std  — train {y_train.std():,.1f} | test {y_test.std():,.1f}")

# Reused everywhere so that every CV number in this notebook is computed identically.
CV = KFold(n_splits=5, shuffle=True, random_state=SEED)

**Read this:** 800 training rows, 200 test rows. Train and test means are close, so the
random split didn't hand us an unrepresentative test set. With only 200 test rows, small
RMSE differences between models are inside the noise — which is why the evaluation table
below carries cross-validated scores with standard deviations alongside the single test
number.

## 4. Baselines

Two floors that every later model must clear:

- **Mean baseline** — always predict the training mean. R² = 0 by construction. If a
  model can't beat this it has learned nothing at all.
- **`sales`-only linear regression** — the real bar. Given what the correlation matrix
  showed, this simple model should already be close to the ceiling. Any complex model
  that fails to beat it has failed, regardless of how good its absolute R² looks.

In [ ]:
def evaluate(model, X_te, y_te, name):
    """Return the four test metrics as a dict. MAPE is reported as a percentage."""
    pred = model.predict(X_te)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_te, pred),
        "RMSE": np.sqrt(mean_squared_error(y_te, pred)),  # sqrt() by hand: `squared=False` is gone in sklearn >=1.6
        "R2": r2_score(y_te, pred),
        "MAPE_%": mean_absolute_percentage_error(y_te, pred) * 100,
    }


results = []

# Baseline (a): the training mean.
mean_base = DummyRegressor(strategy="mean").fit(X_train, y_train)
results.append(evaluate(mean_base, X_test, y_test, "Baseline: mean"))

# Baseline (b): one feature, one coefficient.
sales_base = Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())])
sales_base.fit(X_train[["sales"]], y_train)
sales_pred = sales_base.predict(X_test[["sales"]])
results.append(
    {
        "Model": "Baseline: sales-only LR",
        "MAE": mean_absolute_error(y_test, sales_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, sales_pred)),
        "R2": r2_score(y_test, sales_pred),
        "MAPE_%": mean_absolute_percentage_error(y_test, sales_pred) * 100,
    }
)

display(pd.DataFrame(results).set_index("Model"))

**Read this:** The mean baseline lands at R² = 0, as it must. The `sales`-only
regression is already at ~0.97. That number is the bar. Everything in the next section
is competing for the last few percent, and the honest framing of any later result is
"beat a one-variable model by X", not "achieved R² of 0.97".

> **Methodological note on MAPE.** You asked for it and it's included, but MAPE is
> asymmetric — it penalises over-prediction more heavily than under-prediction of the
> same absolute size, and it explodes when the target approaches zero. Here the target is
> comfortably far from zero so it behaves, but I'd lead with RMSE and MAE when reporting.

## 5. Models

Five candidates, each wrapped in a pipeline so scaling is refit inside every CV fold
rather than once on all the training data.

Scaling is strictly necessary only for Ridge and Lasso — their penalties are applied to
raw coefficient magnitudes, so unscaled features are penalised in proportion to their
units rather than their importance. Trees are scale-invariant and don't need it. It is
applied uniformly anyway because a mixed pipeline is a maintenance trap, and it costs the
tree models nothing.

In [ ]:
alpha_grid = {"model__alpha": np.logspace(-3, 3, 25)}

models = {
    "Linear Regression": Pipeline(
        [("scale", StandardScaler()), ("model", LinearRegression())]
    ),
    "Ridge (tuned)": GridSearchCV(
        Pipeline([("scale", StandardScaler()), ("model", Ridge(random_state=SEED))]),
        param_grid=alpha_grid,
        cv=CV,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    ),
    "Lasso (tuned)": GridSearchCV(
        Pipeline(
            [("scale", StandardScaler()), ("model", Lasso(random_state=SEED, max_iter=20_000))]
        ),
        param_grid=alpha_grid,
        cv=CV,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    ),
    "Random Forest": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=400, min_samples_leaf=2, random_state=SEED, n_jobs=-1
                ),
            ),
        ]
    ),
    "Gradient Boosting": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                GradientBoostingRegressor(
                    n_estimators=400, learning_rate=0.05, max_depth=3, random_state=SEED
                ),
            ),
        ]
    ),
}

fitted = {}
for name, est in models.items():
    est.fit(X_train, y_train)
    fitted[name] = est
    if isinstance(est, GridSearchCV):
        print(f"{name:<20} best alpha = {est.best_params_['model__alpha']:.4g}")
    else:
        print(f"{name:<20} fitted")

**Read this:** Note the alpha values chosen for Ridge and Lasso. A very small alpha means
the penalty is doing almost nothing and the model has collapsed back toward plain
least squares — which is the correct behaviour when there is no multicollinearity to
regularise away. A large Lasso alpha would be informative in the other direction: it
would mean the model is actively zeroing out the noise features.

## 6. Evaluation

Test-set metrics plus 5-fold cross-validated RMSE with its standard deviation. The CV
column is the more trustworthy of the two: it averages over five different splits
instead of betting everything on one arbitrary 200-row sample.

In [ ]:
rows = list(results)  # carry the two baselines into the same table

for name, est in fitted.items():
    row = evaluate(est, X_test, y_test, name)
    cv_rmse = -cross_val_score(
        est, X_train, y_train, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=-1
    )
    row["CV_RMSE"] = cv_rmse.mean()
    row["CV_RMSE_std"] = cv_rmse.std()
    rows.append(row)

comparison = (
    pd.DataFrame(rows)
    .set_index("Model")
    .sort_values("RMSE")[["MAE", "RMSE", "R2", "MAPE_%", "CV_RMSE", "CV_RMSE_std"]]
)
display(comparison)

best_name = comparison.index[0]
best_model = fitted.get(best_name)
print(f"\nBest by test RMSE: {best_name}")

**Read this:** The linear family clusters at the top and the tree ensembles trail. That
is the expected outcome for a relationship this linear — trees approximate a straight
line with a staircase of splits, which costs accuracy rather than buying it.

Now the honest part. The winning R² is high, but compare it to the `sales`-only baseline
a few rows down the same table: the gap is small. **One feature is doing nearly all the
work.** Section 9 quantifies exactly how small the remainder is.

Also note `CV_RMSE_std`: the fold-to-fold variation is comparable to the gaps between
the top few models. Those models are, statistically, tied.

> **Methodological note on selecting by test RMSE.** You asked to sort by test RMSE and
> pick the winner from it, and that's what the code does. Strictly, choosing among six
> models by their test scores turns the test set into a selection set, and the winner's
> reported test metric is then optimistically biased — the test set is no longer a clean
> held-out estimate. The rigorous procedure is to select on `CV_RMSE` and touch the test
> set exactly once, for the chosen model. In this particular case the two orderings agree,
> so nothing downstream changes; on a harder dataset it would.

## 7. Diagnostics for the best model

Metrics compress everything into one number and hide the shape of the errors. These three
plots are where systematic problems become visible.

In [ ]:
best_pred = best_model.predict(X_test)
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

lo = min(y_test.min(), best_pred.min())
hi = max(y_test.max(), best_pred.max())
axes[0].scatter(y_test, best_pred, alpha=0.45, s=22, edgecolor="none")
axes[0].plot([lo, hi], [lo, hi], "--", color="#C44E52", lw=1.8, label="perfect prediction")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Predicted vs actual — {best_name}")
axes[0].legend(frameon=False)

axes[1].scatter(best_pred, residuals, alpha=0.45, s=22, edgecolor="none")
axes[1].axhline(0, ls="--", color="#C44E52", lw=1.8)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Residuals vs predicted")

axes[2].hist(residuals, bins=30, color="#4C72B0", edgecolor="white")
axes[2].axvline(0, ls="--", color="#C44E52", lw=1.8)
axes[2].set_xlabel("Residual")
axes[2].set_ylabel("Count")
axes[2].set_title("Residual distribution")

plt.tight_layout()
plt.show()

print(f"Residual mean    : {residuals.mean():,.2f}   (want ~0 — non-zero means systematic bias)")
print(f"Residual std dev : {residuals.std():,.2f}")
print(f"Residual skew    : {pd.Series(residuals).skew():.3f}   (want ~0)")

# Heteroscedasticity check without extra dependencies: does error size grow with
# prediction size? Split the test set at the median prediction and compare spread.
lower_half = residuals[best_pred <= np.median(best_pred)]
upper_half = residuals[best_pred > np.median(best_pred)]
spread_ratio = upper_half.std() / lower_half.std()
print(f"\nResidual std, lower half of predictions: {lower_half.std():,.2f}")
print(f"Residual std, upper half of predictions: {upper_half.std():,.2f}")
print(f"Ratio: {spread_ratio:.3f}   (near 1.0 = homoscedastic; >1.5 or <0.67 = a problem)")

**Read this:**

- **Predicted vs actual** — points hug the 45° line across the full range, with no
  fan-out at the extremes and no curvature. The model is not systematically
  over-predicting small values or under-predicting large ones.
- **Residuals vs predicted** — a formless horizontal band centred on zero. This is what
  you want. A funnel shape would mean heteroscedasticity (and would invalidate the
  constant-width prediction interval built in section 10); a curve would mean an
  un-modelled non-linearity.
- **Residual histogram** — roughly symmetric and centred at zero, near-normal. No fat
  tail on one side, so no class of rows is being handled badly.
- **The spread ratio** printed above is the numerical version of the funnel check. Close
  to 1.0 means error magnitude does not depend on prediction magnitude.

Judge these on the output you actually get, not on this text — if the ratio comes back
far from 1.0, the prediction interval in section 10 is too narrow at the top end and too
wide at the bottom.

## 8. Feature importance

Two independent views, because each has a known failure mode.

**Permutation importance** is computed on the *test* set: shuffle one column, measure how
much test R² degrades. Computing it on training data instead would reward memorisation.
This is deliberately not `.feature_importances_` from the tree — impurity-based importance
is biased toward high-cardinality continuous features and will assign non-trivial
importance to pure noise columns simply because they offer many split points.

**Standardised linear coefficients** give a second opinion from a completely different
mechanism. When two unrelated methods agree, the conclusion is solid.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=30, random_state=SEED, scoring="r2", n_jobs=-1
)

perm_df = (
    pd.DataFrame(
        {"Feature": FEATURES, "R2_drop": perm.importances_mean, "Std": perm.importances_std}
    )
    .sort_values("R2_drop", ascending=False)
    .reset_index(drop=True)
)
display(perm_df)

plt.figure(figsize=(8, 4.5))
plt.barh(perm_df["Feature"], perm_df["R2_drop"], xerr=perm_df["Std"], color="#4C72B0")
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Drop in test R² when the feature is shuffled")
plt.title("Permutation importance (test set, 30 repeats)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Second opinion: coefficients on standardised features are directly comparable to each
# other because every feature has been put on a unit-variance scale.
coef_pipe = Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())]).fit(
    X_train, y_train
)
coefs = (
    pd.DataFrame(
        {"Feature": FEATURES, "Std_coefficient": coef_pipe.named_steps["lr"].coef_}
    )
    .assign(Abs=lambda d: d["Std_coefficient"].abs())
    .sort_values("Abs", ascending=False)
    .drop(columns="Abs")
    .reset_index(drop=True)
)
display(coefs)

signal = perm_df[perm_df["R2_drop"] > 2 * perm_df["Std"]]["Feature"].tolist()
noise = [f for f in FEATURES if f not in signal]
print(f"Carries real signal (importance > 2 std errors above zero): {signal}")
print(f"Indistinguishable from noise                              : {noise}")

**Read this:** Both methods point the same way. `sales` dominates by orders of magnitude.
The five macro indicators produce permutation importances at or indistinguishable from
zero. Two patterns are possible here and both mean the same thing:

- **Exactly zero** — if the winning model is Lasso, the L1 penalty has already driven
  those coefficients to 0, so shuffling a column the model never reads changes nothing.
  The regulariser identified them as noise without being told to.
- **Small or negative** — for the unpenalised models, scoring slightly *better* with a
  column randomised is the clearest possible evidence that the feature contributes noise.

The standardised coefficients corroborate it: `sales` carries a large coefficient, the
macro features carry coefficients small relative to their own uncertainty.

**Conclusion: one real driver, five passengers.** If this were a production model, the
five macro features should be dropped — they add data-collection cost, dependency risk,
and failure modes while contributing nothing.

## 9. Ablation

The direct test. Refit the winning model twice — once on `sales` alone, once on
everything — and read the difference. This settles the question that importance plots
only gesture at.

In [ ]:
def clone_and_fit(template, cols):
    """Refit a fresh copy of the winning estimator on a subset of columns."""
    from sklearn.base import clone

    est = clone(template)
    est.fit(X_train[cols], y_train)
    return est


ablation_rows = []
for label, cols in [("sales only", ["sales"]), ("all 6 features", FEATURES)]:
    est = clone_and_fit(best_model, cols)
    pred = est.predict(X_test[cols])
    ablation_rows.append(
        {
            "Feature set": label,
            "n_features": len(cols),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
            "R2": r2_score(y_test, pred),
        }
    )

ablation = pd.DataFrame(ablation_rows).set_index("Feature set")
display(ablation)

r2_solo = ablation.loc["sales only", "R2"]
r2_full = ablation.loc["all 6 features", "R2"]
delta_r2 = r2_full - r2_solo
delta_pct = delta_r2 / r2_solo * 100

print(f"R² using sales alone      : {r2_solo:.5f}")
print(f"R² using all six features : {r2_full:.5f}")
print(f"Absolute gain             : {delta_r2:+.5f}")
print(f"Relative gain             : {delta_pct:+.3f}%")

if abs(delta_pct) < 1.0:
    print(
        "\nVERDICT: the five macro features add less than 1% to R². They are, for "
        "practical purposes, worthless. This model is a one-variable model wearing a "
        "six-variable costume."
    )
else:
    print(f"\nVERDICT: the macro features contribute {delta_pct:.2f}% — non-trivial, worth keeping.")

**Read this:** This is the number to quote if anyone asks what the macro indicators are
worth. Whatever the headline R² is, the honest description of this model is: *it reads
the `sales` column and scales it.* The five macro features are, on this data, an
expensive way to add zero.

A practical consequence: deploy the `sales`-only model. It is simpler, faster, needs one
input instead of six, and cannot break when a macro data feed goes down.

## 10. Prediction function with intervals

A point estimate on its own is a false promise of precision. The function below returns
a central prediction plus a 90% interval.

The interval width comes from **cross-validated residuals on the training set**, not from
training-fit residuals. In-sample residuals are systematically too small — the model has
already seen those rows — so an interval built from them would be too narrow and would
fail to cover 90% of real cases.

In [ ]:
# Honest residual distribution: each prediction here comes from a fold that excluded the
# row being predicted.
cv_preds = cross_val_predict(best_model, X_train, y_train, cv=CV, n_jobs=-1)
cv_residuals = y_train.values - cv_preds

LOWER_Q, UPPER_Q = np.percentile(cv_residuals, [5, 95])
print(f"90% residual band from CV: [{LOWER_Q:,.1f}, {UPPER_Q:,.1f}]")
print(f"Interval width: {UPPER_Q - LOWER_Q:,.1f}")


def predict_target_sales(new_data: pd.DataFrame, with_interval: bool = True):
    """Predict target_sales for new rows.

    Validates columns up front rather than letting sklearn fail with an opaque shape
    error twelve frames deep. Reorders columns to training order, because a silently
    mis-ordered DataFrame produces confident nonsense rather than an exception.

    Returns an ndarray of point predictions, or a DataFrame with 90% bounds.
    """
    if not isinstance(new_data, pd.DataFrame):
        raise TypeError("new_data must be a pandas DataFrame")

    missing_cols = set(FEATURES) - set(new_data.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

    X_new = new_data[FEATURES].copy()  # reorder to match training
    point = best_model.predict(X_new)

    if not with_interval:
        return point

    return pd.DataFrame(
        {
            "prediction": point,
            "lower_90": point + LOWER_Q,
            "upper_90": point + UPPER_Q,
        },
        index=new_data.index,
    )


# Three synthetic rows spanning low / typical / high sales, with macro values held near
# their means so the sales effect is isolated.
example = pd.DataFrame(
    {
        "sales": [3_500.0, 5_000.0, 7_000.0],
        "market_indicator_1": [200.0, 201.4, 205.0],
        "market_indicator_2": [48.0, 50.0, 52.0],
        "gdp_growth": [2.5, 3.0, 3.5],
        "unemployment_rate": [5.5, 4.95, 4.2],
        "inflation_rate": [1.8, 1.98, 2.3],
    },
    index=["low_sales", "typical_sales", "high_sales"],
)

display(example)
display(predict_target_sales(example))

**Read this:** The predictions scale with `sales` almost exactly as the correlation
implied. The interval is the useful part — quote the range, not the midpoint.

> **Methodological note on the interval.** This is a constant-width band: every
> prediction gets the same ± regardless of its magnitude. That is only valid because the
> heteroscedasticity check in section 7 came back clean. If that ratio had been far from
> 1.0, this interval would be wrong in a direction that varies by row, and you'd want
> quantile regression or a conformal method instead. Also note this interval covers
> *residual* uncertainty only — it does not account for uncertainty in the fitted
> parameters themselves, so it is mildly optimistic.

## 11. Save and reload

`joblib.dump` on the whole pipeline, not just the estimator. Saving a bare model and
forgetting its scaler is a classic production failure: the reloaded model receives raw
unscaled inputs and returns confident garbage with no error raised.

In [ ]:
MODEL_PATH = "target_sales_model.joblib"

joblib.dump(
    {
        "pipeline": best_model,
        "features": FEATURES,
        "target": TARGET,
        "model_name": best_name,
        "residual_bounds_90": (float(LOWER_Q), float(UPPER_Q)),
    },
    MODEL_PATH,
)
print(f"Saved to {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:,.1f} KB)")

# Prove the round-trip is lossless before trusting it.
bundle = joblib.load(MODEL_PATH)
reloaded = bundle["pipeline"]

original_pred = best_model.predict(X_test)
reloaded_pred = reloaded.predict(X_test)

print(f"\nReloaded model : {bundle['model_name']}")
print(f"Expects columns: {bundle['features']}")
print(f"Predictions identical: {np.allclose(original_pred, reloaded_pred)}")
print(f"Max difference       : {np.abs(original_pred - reloaded_pred).max():.2e}")

**Read this:** Identical predictions after the round-trip, so the artefact is safe to
deploy. The bundle stores the feature list and interval bounds alongside the pipeline —
a model file that doesn't record what inputs it expects is a bug waiting to happen.

In Colab, download it with:
```python
from google.colab import files
files.download("target_sales_model.joblib")
```
(Left commented out so this notebook runs cleanly outside Colab.)

## 12. Conclusion

In [ ]:
print("=" * 68)
print("SUMMARY")
print("=" * 68)
print(f"Best model         : {best_name}")
print(f"Test RMSE          : {comparison.loc[best_name, 'RMSE']:,.2f}")
print(f"Test MAE           : {comparison.loc[best_name, 'MAE']:,.2f}")
print(f"Test R²            : {comparison.loc[best_name, 'R2']:.4f}")
print(f"Test MAPE          : {comparison.loc[best_name, 'MAPE_%']:.2f}%")
print(f"CV RMSE            : {comparison.loc[best_name, 'CV_RMSE']:,.2f} "
      f"(+/- {comparison.loc[best_name, 'CV_RMSE_std']:,.2f})")
print("-" * 68)
print(f"R², sales only     : {r2_solo:.5f}")
print(f"R², all features   : {r2_full:.5f}")
print(f"Value of the other five features: {delta_pct:+.3f}% relative R²")
print("=" * 68)

### What was found

**Best model.** The linear family wins; see the summary block above for exact figures.
The relationship is genuinely linear, so tree ensembles add complexity without accuracy.

**What actually drives the target.** `sales`, and effectively nothing else. It correlates
at ~0.985 and survives every robustness check. The five macro indicators
(`market_indicator_1`, `market_indicator_2`, `gdp_growth`, `unemployment_rate`,
`inflation_rate`) fail all three tests independently: near-zero raw correlation,
permutation importance indistinguishable from zero (some negative), and under 1% relative
R² contribution in ablation.

**The headline number in context.** The high R² is real but it belongs to one feature.
Reporting "our model explains 97% of variance" without adding "using a single input that
already correlated at 0.985" would be a material omission. The modelling contributed
very little on top of that correlation.

### Limitations — read before using any of this

1. **This is not a forecast.** There is no time column. The model answers "given this
   row's `sales`, what is its `target_sales`?" — not "what will sales be next quarter?"
   Genuine forecasting needs time-ordered observations, a chronological (never random)
   train/test split, and time-series validation. Do not describe this output as a
   prediction of the future; it is a conditional estimate given inputs you must already
   possess.
2. **The data is simulated.** Symmetric distributions, zero missingness, no duplicates
   and no outliers do not occur in real financial data. Performance here says nothing
   about performance on production data.
3. **Correlation, not causation.** Nothing here establishes that `sales` *causes*
   `target_sales`. Given the naming and the strength of the relationship, the likelier
   explanation is that the two are definitionally linked in whatever process generated
   the file — possibly `target_sales` was constructed from `sales`. If so, the model is
   recovering an arithmetic identity, not learning economics.
4. **Small test set.** 200 rows. Differences of a few RMSE points between the top models
   are inside the fold-to-fold noise; treat them as tied.
5. **Selection on the test set.** Six models were compared on the same held-out rows, so
   the winner's test metric is mildly optimistic. The CV column is the safer estimate.
6. **Interval caveats.** The 90% band is constant-width and covers residual scatter only,
   excluding parameter uncertainty. It is valid only while residuals stay homoscedastic.
7. **No drift monitoring.** If deployed, the relationship must be re-checked on fresh
   data on a schedule. A model this dependent on one input fails hard and silently if
   that input's meaning or scale ever changes upstream.

### Recommended next step

Deploy the `sales`-only model, not the six-feature one. It performs the same, costs less
to run, and has five fewer ways to break. Then go and find the time-ordered source data
if an actual forecast is what's needed.